# 05 · Automatización de copias de seguridad

Laboratorio detallado: generar versiones completas, verificar bytes y recuperar datos útiles. **Todos los archivos son ficticios y se crean dentro de un contenedor temporal propio.** El motor no cifra ni implementa 3-2-1. No instala tareas.

## Requisitos y recorrido

Python 3.11+ y biblioteca estándar. Abre Jupyter desde la carpeta del bloque para encontrar `copias_seguridad.py`. Ejecuta desde un kernel limpio. Las definiciones se desarrollan aquí para leerlas; el ensayo CLI utiliza además el archivo entregado.

Primero estudia contratos y funciones; después ejecuta los doce ejercicios independientes. Una celda que crea una carpeta nueva no debe repetirse sin reiniciar el laboratorio. Las salidas del solucionario son evidencia de una ejecución, no rutas para copiar en tu equipo.

In [ ]:
"""Motor docente de copias completas versionadas, Python 3.11+.

Solo opera sobre un proyecto sintético creado por crear-demo. No es una
solución 3-2-1, no cifra y no conserva ACL, enlaces ni carpetas vacías.
Códigos: 0 correcto, 10 integridad, 20 contrato, 30 operativo, 40 bloqueo.
"""
from pathlib import Path, PurePosixPath, PureWindowsPath
from contextlib import contextmanager
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
import argparse
import hashlib
import json
import logging
import os
import re
import shutil
import sys
import time
import uuid

VERSION = "copias-aula-1.0"
MAX_ARCHIVOS = 1000
MAX_ARCHIVO = 10 * 1024**2
MAX_TOTAL = 50 * 1024**2
BLOQUE = 64 * 1024
PATRON_COPIA = r"copia_[0-9]{8}T[0-9]{6}Z_[0-9a-f]{8}"
LOG = logging.getLogger("copias")



import csv
import subprocess
PROYECTO = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
MOTOR = PROYECTO / "copias_seguridad.py"
if not MOTOR.is_file():
    raise FileNotFoundError("Abre Jupyter desde la carpeta del bloque")


## ContratoError

Tipo explícito de error para separar fallos de contrato, integridad y bloqueo. Permite al CLI devolver una categoría útil sin depender de analizar texto.

In [ ]:
class ContratoError(ValueError):
    pass


## IntegridadError

Tipo explícito de error para separar fallos de contrato, integridad y bloqueo. Permite al CLI devolver una categoría útil sin depender de analizar texto.

In [ ]:
class IntegridadError(ValueError):
    pass


## BloqueoOcupado

Tipo explícito de error para separar fallos de contrato, integridad y bloqueo. Permite al CLI devolver una categoría útil sin depender de analizar texto.

In [ ]:
class BloqueoOcupado(RuntimeError):
    pass


## json_estricto

Rechazar claves duplicadas evita interpretar el mismo manifiesto de dos maneras. NaN tampoco forma parte del contrato. Un JSON legible todavía debe pasar la validación de campos.

In [ ]:
def json_estricto(texto):
    def pares(p):
        d = {}
        for k, v in p:
            if k in d:
                raise ValueError("Clave JSON repetida")
            d[k] = v
        return d
    def no_constante(v):
        raise ValueError("Constante JSON no admitida: " + v)
    return json.loads(texto, object_pairs_hook=pares, parse_constant=no_constante)


## nombre_simple

Un destino es un solo componente. Se rechazan separadores, controles y nombres reservados en Windows. Se limita la práctica a nombres portables, sin asumir equivalencia universal entre sistemas de archivos.

In [ ]:
def nombre_simple(nombre):
    if not isinstance(nombre, str) or not nombre or nombre in {".", ".."}:
        raise ContratoError("Nombre vacío o reservado")
    if nombre != nombre.strip() or nombre.endswith("."):
        raise ContratoError("Espacios extremos o punto final no admitidos")
    if any(c in '<>:"/\\|?*' or ord(c) < 32 for c in nombre):
        raise ContratoError("Nombre con separadores o caracteres no admitidos")
    reservados = {"CON", "PRN", "AUX", "NUL"} | {f"{p}{i}" for p in ("COM", "LPT") for i in range(1, 10)}
    if nombre.split(".", 1)[0].upper() in reservados:
        raise ContratoError("Nombre reservado en Windows")
    return nombre


## ruta_relativa_segura

El manifiesto almacena rutas POSIX relativas y normalizadas. Se rechazan ascensos, rutas absolutas y componentes peligrosos antes de construir una ruta física.

In [ ]:
def ruta_relativa_segura(texto):
    if not isinstance(texto, str) or not texto or "\\" in texto:
        raise IntegridadError("Ruta relativa incompatible")
    p = PurePosixPath(texto)
    if not p.parts or p.is_absolute() or PureWindowsPath(texto).anchor or p.as_posix() != texto:
        raise IntegridadError("Ruta absoluta o no canónica")
    try:
        for parte in p.parts:
            nombre_simple(parte)
    except ContratoError as error:
        raise IntegridadError("Componente de ruta incompatible") from error
    return p


## cargar_proyecto

El marcador identifica un proyecto de aula y exige carpetas reales. Evita errores de selección; no crea un sandbox ni protege contra un atacante que cambie el árbol concurrentemente.

In [ ]:
def cargar_proyecto(proyecto):
    original = Path(proyecto).absolute()
    if original.is_symlink():
        raise ContratoError("La raíz no puede ser un enlace")
    try:
        raiz = original.resolve(strict=True)
        marca = raiz / "proyecto_backup.json"
        if marca.is_symlink() or not marca.is_file() or marca.stat().st_size > 4096:
            raise ContratoError("No se reconoce el proyecto de aula")
        datos = json_estricto(marca.read_text(encoding="utf-8"))
    except (FileNotFoundError, NotADirectoryError, ValueError, UnicodeError) as error:
        raise ContratoError("Proyecto de aula ausente o incompatible") from error
    if not isinstance(datos, dict) or datos.get("tipo") != "demo_copias" or type(datos.get("schema")) is not int or datos["schema"] != 1:
        raise ContratoError("Marcador de proyecto incompatible")
    for nombre in ("origen", "copias", "restauraciones", "estado"):
        ruta = raiz / nombre
        if ruta.is_symlink() or not ruta.is_dir():
            raise ContratoError("Falta una carpeta real del proyecto: " + nombre)
    return raiz


## enumerar_archivos

El recorrido explicita cada error de lectura y rechaza enlaces y recursos especiales. Una carpeta vacía no es un archivo y queda fuera de este contrato.

In [ ]:
def enumerar_archivos(carpeta):
    """Árbol privado y estable; rechaza enlaces y recursos no regulares."""
    carpeta = Path(carpeta)
    if carpeta.is_symlink() or not carpeta.is_dir():
        raise IntegridadError("Se requiere una carpeta real")
    archivos, pendientes = [], [carpeta]
    while pendientes:
        actual = pendientes.pop()
        for p in sorted(actual.iterdir(), key=lambda x: x.name):
            if p.is_symlink():
                raise IntegridadError("No se admiten enlaces")
            if p.is_dir():
                pendientes.append(p)
            elif p.is_file():
                archivos.append(p)
            else:
                raise IntegridadError("Recurso distinto de archivo regular")
            if len(archivos) > MAX_ARCHIVOS:
                raise IntegridadError("Demasiados archivos para el laboratorio")
    return sorted(archivos, key=lambda p: p.relative_to(carpeta).as_posix())


## hash_archivo

Lee en bloques, cuenta los bytes realmente leídos y calcula SHA-256. El espacio auxiliar del hash queda acotado. La huella no cifra el contenido ni autentica al autor.

In [ ]:
def hash_archivo(ruta):
    digest, cantidad = hashlib.sha256(), 0
    with Path(ruta).open("rb") as f:
        while bloque := f.read(BLOQUE):
            cantidad += len(bloque)
            if cantidad > MAX_ARCHIVO:
                raise IntegridadError("Archivo superior al límite de aula")
            digest.update(bloque)
    return cantidad, digest.hexdigest()


## inventariar

Ordena rutas y construye registros de tamaño y huella. Aplica límites por archivo y total, y rechaza colisiones al ignorar mayúsculas y conflictos archivo/carpeta.

In [ ]:
def inventariar(carpeta):
    carpeta = Path(carpeta)
    filas, nombres, total = [], set(), 0
    for p in enumerar_archivos(carpeta):
        relativa = p.relative_to(carpeta).as_posix()
        ruta_relativa_segura(relativa)
        clave = relativa.casefold()
        if clave in nombres or any(clave.startswith(v + "/") or v.startswith(clave + "/") for v in nombres):
            raise IntegridadError("Colisión de nombres o carpetas al ignorar mayúsculas")
        nombres.add(clave)
        n, sha = hash_archivo(p)
        total += n
        if total > MAX_TOTAL:
            raise IntegridadError("Tamaño total superior al límite de aula")
        filas.append({"ruta": relativa, "bytes": n, "sha256": sha})
    return filas


## validar_manifiesto

Valida campos exactos, tipos, identidad, UTC, rutas, tamaños, huellas y sumas. Los booleanos se excluyen de los enteros válidos. No se usan rutas de un manifiesto antes de validarlas.

In [ ]:
def validar_manifiesto(datos):
    claves = {"schema", "id", "creada_utc", "tipo", "motor", "archivos", "bytes_totales"}
    if not isinstance(datos, dict) or set(datos) != claves or type(datos["schema"]) is not int or datos["schema"] != 1:
        raise IntegridadError("Esquema de manifiesto incompatible")
    if not isinstance(datos["id"], str) or re.fullmatch(PATRON_COPIA, datos["id"]) is None:
        raise IntegridadError("Identidad de copia incompatible")
    if datos["tipo"] != "completa" or datos["motor"] != VERSION:
        raise IntegridadError("Tipo o versión no compatibles con este motor")
    try:
        fecha = datetime.fromisoformat(datos["creada_utc"])
        if fecha.utcoffset() != timezone.utc.utcoffset(fecha):
            raise ValueError("Se requiere UTC")
    except (TypeError, ValueError, AttributeError) as error:
        raise IntegridadError("Fecha UTC incompatible") from error
    if not isinstance(datos["archivos"], list) or len(datos["archivos"]) > MAX_ARCHIVOS:
        raise IntegridadError("Lista de archivos incompatible")
    vistos, total = set(), 0
    for fila in datos["archivos"]:
        if not isinstance(fila, dict) or set(fila) != {"ruta", "bytes", "sha256"}:
            raise IntegridadError("Entrada de manifiesto incompatible")
        p = ruta_relativa_segura(fila["ruta"])
        clave = p.as_posix().casefold()
        if clave in vistos or any(clave.startswith(v + "/") or v.startswith(clave + "/") for v in vistos):
            raise IntegridadError("Ruta duplicada o incompatible con otra")
        vistos.add(clave)
        if type(fila["bytes"]) is not int or not 0 <= fila["bytes"] <= MAX_ARCHIVO:
            raise IntegridadError("Tamaño de archivo incompatible")
        if not isinstance(fila["sha256"], str) or re.fullmatch(r"[0-9a-f]{64}", fila["sha256"]) is None:
            raise IntegridadError("Huella incompatible")
        total += fila["bytes"]
    if total > MAX_TOTAL or type(datos["bytes_totales"]) is not int or total != datos["bytes_totales"]:
        raise IntegridadError("El total del manifiesto no reconcilia")
    return datos


## verificar_directorio

Contrasta el conjunto real con el declarado: ni faltantes ni adicionales. Un archivo de igual tamaño pero distintos bytes también falla. Si un atacante modifica manifiesto y contenido de forma coherente, esta comprobación no demuestra autenticidad.

In [ ]:
def verificar_directorio(carpeta):
    carpeta = Path(carpeta)
    if carpeta.is_symlink() or not carpeta.is_dir():
        raise IntegridadError("Copia no disponible como carpeta real")
    if {p.name for p in carpeta.iterdir()} != {"manifiesto.json", "contenido"}:
        raise IntegridadError("Estructura de copia inesperada")
    archivo = carpeta / "manifiesto.json"
    if archivo.is_symlink() or not archivo.is_file() or archivo.stat().st_size > 2 * 1024**2:
        raise IntegridadError("Manifiesto ausente o demasiado grande")
    try:
        datos = validar_manifiesto(json_estricto(archivo.read_text(encoding="utf-8")))
    except (ValueError, UnicodeError) as error:
        raise IntegridadError("Manifiesto inválido: " + str(error)) from error
    actual = inventariar(carpeta / "contenido")
    if sorted(actual, key=lambda f: f["ruta"]) != sorted(datos["archivos"], key=lambda f: f["ruta"]):
        raise IntegridadError("Rutas, tamaños o huellas no coinciden")
    return datos


## localizar_copia

Solo reconoce identidades de generación publicadas y exige coincidencia entre carpeta e ID del manifiesto. Los parciales no se aceptan como versiones recuperables.

In [ ]:
def localizar_copia(raiz, nombre):
    if not isinstance(nombre, str) or re.fullmatch(PATRON_COPIA, nombre) is None:
        raise ContratoError("Nombre de copia publicada incompatible")
    ruta = raiz / "copias" / nombre
    datos = verificar_directorio(ruta)
    if datos["id"] != nombre:
        raise IntegridadError("La identidad del manifiesto no coincide con la carpeta")
    return ruta, datos


## bloqueo

Creación exclusiva con token propio para escritores cooperantes. Un cierre forzado puede dejar bloqueo huérfano. No se retira automáticamente por edad o PID y no es un bloqueo distribuido.

In [ ]:
@contextmanager
def bloqueo(raiz):
    archivo = raiz / "estado" / "copias.lock"
    token = uuid.uuid4().hex
    try:
        fd = os.open(archivo, os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o600)
    except FileExistsError as error:
        raise BloqueoOcupado("Ya existe una ejecución o un bloqueo pendiente de revisión") from error
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump({"token": token, "pid": os.getpid()}, f)
        yield
    finally:
        if archivo.exists() and not archivo.is_symlink():
            try:
                if json_estricto(archivo.read_text()).get("token") == token:
                    archivo.unlink()
            except (OSError, ValueError, AttributeError):
                LOG.error("Revisar bloqueo: no se pudo retirar con su identidad")


## copiar_contraste

La copia calcula huella durante la transferencia y la compara con el inventario. Detecta cambios ordinarios entre ambas lecturas, pero no garantiza una transacción entre varios archivos: el origen debe estar en reposo.

In [ ]:
def copiar_contraste(origen, destino, esperado):
    """Crea exclusivamente; fuente quiescente y árbol privado como supuestos."""
    destino.parent.mkdir(parents=True, exist_ok=True)
    digest, n = hashlib.sha256(), 0
    with origen.open("rb") as fuente, destino.open("xb") as salida:
        while bloque := fuente.read(BLOQUE):
            n += len(bloque)
            if n > MAX_ARCHIVO:
                raise IntegridadError("Archivo cambió o supera el límite")
            digest.update(bloque)
            salida.write(bloque)
    if n != esperado["bytes"] or digest.hexdigest() != esperado["sha256"]:
        raise IntegridadError("Contenido cambió entre inventario y copia")


## crear_copia

Sin aplicar, solo inventaría. Al aplicar, bloquea, crea etapa, copia, escribe manifiesto, verifica y publica mediante renombrado. Un fallo puede dejar un parcial para revisión. El renombrado no constituye un snapshot del origen ni garantía ante cualquier corte eléctrico.

In [ ]:
def crear_copia(proyecto, aplicar=False):
    raiz = cargar_proyecto(proyecto)
    if type(aplicar) is not bool:
        raise ContratoError("aplicar debe ser booleano")
    def preparar():
        filas = inventariar(raiz / "origen")
        if not filas:
            raise ContratoError("No se publica una copia vacía en esta práctica")
        return filas
    if not aplicar:
        filas = preparar()
        return {"accion": "crear", "aplicado": False, "n_archivos": len(filas),
                "bytes_totales": sum(f["bytes"] for f in filas)}
    with bloqueo(raiz):
        filas = preparar()
        instante = datetime.now(timezone.utc)
        identificador = "copia_" + instante.strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8]
        parcial = raiz / "copias" / (".parcial_" + uuid.uuid4().hex)
        parcial.mkdir(mode=0o700)
        (parcial / "contenido").mkdir()
        for fila in filas:
            partes = PurePosixPath(fila["ruta"]).parts
            copiar_contraste((raiz / "origen").joinpath(*partes),
                            (parcial / "contenido").joinpath(*partes), fila)
        manifiesto = {"schema": 1, "id": identificador,
                      "creada_utc": instante.isoformat(), "tipo": "completa",
                      "motor": VERSION, "archivos": filas,
                      "bytes_totales": sum(f["bytes"] for f in filas)}
        with (parcial / "manifiesto.json").open("x", encoding="utf-8", newline="\n") as f:
            json.dump(manifiesto, f, ensure_ascii=False, indent=2)
        verificar_directorio(parcial)
        final = raiz / "copias" / identificador
        if final.exists():
            raise FileExistsError(final)
        parcial.rename(final)
        return {"accion": "crear", "aplicado": True, "copia": identificador,
                "n_archivos": len(filas), "bytes_totales": manifiesto["bytes_totales"],
                "verificada": True}


## verificar_copia

Interfaz de lectura que devuelve conteos después de validar el contenido completo. Una respuesta correcta no prueba que la aplicación de negocio pueda usar los datos.

In [ ]:
def verificar_copia(proyecto, nombre):
    raiz = cargar_proyecto(proyecto)
    _, datos = localizar_copia(raiz, nombre)
    return {"accion": "verificar", "copia": nombre, "verificada": True,
            "n_archivos": len(datos["archivos"]), "bytes_totales": datos["bytes_totales"]}


## restaurar_copia

Exige destino nuevo bajo restauraciones. Verifica antes, copia a etapa y contrasta después. La duración devuelta solo cubre transferencia y contraste del destino: no es el RTO del proceso.

In [ ]:
def restaurar_copia(proyecto, nombre, destino, aplicar=False):
    raiz = cargar_proyecto(proyecto)
    nombre_simple(destino)
    if type(aplicar) is not bool:
        raise ContratoError("aplicar debe ser booleano")
    carpeta_destino = raiz / "restauraciones" / destino
    if carpeta_destino.exists() or carpeta_destino.is_symlink():
        raise ContratoError("El destino de restauración debe ser nuevo")
    if not aplicar:
        _, datos = localizar_copia(raiz, nombre)
        return {"accion": "restaurar", "aplicado": False, "destino": destino,
                "n_archivos": len(datos["archivos"])}
    with bloqueo(raiz):
        copia, datos = localizar_copia(raiz, nombre)
        etapa = raiz / "restauraciones" / (".parcial_" + uuid.uuid4().hex)
        etapa.mkdir(mode=0o700)
        inicio = time.perf_counter()
        for fila in datos["archivos"]:
            partes = PurePosixPath(fila["ruta"]).parts
            copiar_contraste((copia / "contenido").joinpath(*partes), etapa.joinpath(*partes), fila)
        actual = inventariar(etapa)
        if actual != sorted(datos["archivos"], key=lambda f: f["ruta"]):
            raise IntegridadError("La restauración no coincide")
        if carpeta_destino.exists() or carpeta_destino.is_symlink():
            raise ContratoError("El destino apareció durante la operación")
        etapa.rename(carpeta_destino)
        return {"accion": "restaurar", "aplicado": True, "destino": destino,
                "n_archivos": len(actual), "bytes_totales": sum(f["bytes"] for f in actual),
                "contenido_verificado": True, "duracion_transferencia_y_contraste_s": time.perf_counter() - inicio}


## retener_copias

Primero valida todas las generaciones reconocidas y calcula el plan. Conserva al menos una, ordenando por UTC declarada. Las carpetas desconocidas y parciales quedan fuera. Un fallo operativo durante varios borrados puede dejar aplicación parcial; no es una transacción de retención.

In [ ]:
def retener_copias(proyecto, conservar=2, aplicar=False):
    raiz = cargar_proyecto(proyecto)
    if type(conservar) is not int or conservar < 1:
        raise ContratoError("Se debe conservar al menos una copia")
    if type(aplicar) is not bool:
        raise ContratoError("aplicar debe ser booleano")
    def planificar():
        candidatas = []
        for p in (raiz / "copias").iterdir():
            if re.fullmatch(PATRON_COPIA, p.name):
                _, datos = localizar_copia(raiz, p.name)
                candidatas.append((datetime.fromisoformat(datos["creada_utc"]), p.name))
        candidatas.sort(reverse=True)
        return {"accion": "retencion", "conservar": [n for _, n in candidatas[:conservar]],
                "retirar": [n for _, n in candidatas[conservar:]], "aplicado": aplicar,
                "criterio": "fecha UTC declarada; integridad comprobada antes de retirar"}
    if not aplicar:
        return planificar()
    with bloqueo(raiz):
        plan = planificar()
        for nombre in plan["retirar"]:
            ruta, _ = localizar_copia(raiz, nombre)
            shutil.rmtree(ruta)  # Solo una generación reconocida y contrastada de este laboratorio.
        return plan


## crear_demo

Crea un proyecto nuevo y privado con cinco archivos sintéticos de 136 bytes y una carpeta vacía. No reutiliza una carpeta existente. Los permisos POSIX no equivalen a una configuración de ACL en Windows.

In [ ]:
def crear_demo(destino):
    raiz = Path(destino).absolute()
    raiz.mkdir(parents=True, exist_ok=False, mode=0o700)
    for carpeta in ("origen", "copias", "restauraciones", "estado"):
        (raiz / carpeta).mkdir()
    marca = {"tipo": "demo_copias", "schema": 1, "id": uuid.uuid4().hex}
    (raiz / "proyecto_backup.json").write_text(json.dumps(marca), encoding="utf-8")
    textos = {"datos/listado_2025.csv": "id;importe\nDEMO-001;1200000\n",
              "contexto/territorios.json": '{"territorio":"DEMO","poblacion":1000}\n',
              "informes/resumen.md": "# Informe ficticio\n\nSolo datos de aula.\n",
              "config/parametros.json": '{"anio":2025,"moneda":"EUR"}\n',
              "vacio.txt": ""}
    for relativa, texto in textos.items():
        p = raiz / "origen" / relativa
        p.parent.mkdir(parents=True, exist_ok=True)
        with p.open("x", encoding="utf-8", newline="\n") as f:
            f.write(texto)
    (raiz / "origen" / "pendiente").mkdir()
    return raiz


## main

CLI no interactiva: JSON en stdout, errores en stderr y códigos separados. El planificador debe distinguir una simulación correcta de una publicación real.

In [ ]:
def main(argv=None):
    for canal in (sys.stdout, sys.stderr):
        if hasattr(canal, "reconfigure"):
            canal.reconfigure(encoding="utf-8")
    parser = argparse.ArgumentParser(description=__doc__)
    sub = parser.add_subparsers(dest="accion", required=True)
    demo = sub.add_parser("crear-demo"); demo.add_argument("destino", type=Path)
    for accion in ("crear", "verificar", "restaurar", "retencion"):
        p = sub.add_parser(accion); p.add_argument("--proyecto", required=True, type=Path)
        if accion in {"verificar", "restaurar"}:
            p.add_argument("--copia", required=True)
        if accion == "restaurar":
            p.add_argument("--destino", required=True)
        if accion == "retencion":
            p.add_argument("--conservar", type=int, default=2)
        if accion != "verificar":
            p.add_argument("--aplicar", action="store_true")
    args = parser.parse_args(argv)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    try:
        if args.accion == "crear-demo":
            resultado = {"proyecto": str(crear_demo(args.destino))}
        elif args.accion == "crear":
            resultado = crear_copia(args.proyecto, args.aplicar)
        elif args.accion == "verificar":
            resultado = verificar_copia(args.proyecto, args.copia)
        elif args.accion == "restaurar":
            resultado = restaurar_copia(args.proyecto, args.copia, args.destino, args.aplicar)
        else:
            resultado = retener_copias(args.proyecto, args.conservar, args.aplicar)
        print(json.dumps(resultado, ensure_ascii=False))
        return 0
    except BloqueoOcupado as error:
        LOG.error("Bloqueo: %s", error); return 40
    except ContratoError as error:
        LOG.error("Contrato: %s", error); return 20
    except IntegridadError as error:
        LOG.error("Integridad: %s", error); return 10
    except OSError:
        LOG.exception("Fallo operativo; revisar parciales"); return 30
    except Exception:
        LOG.exception("Fallo inesperado"); return 30


## Preparar un contenedor propio

Cada ejercicio que escribe usa una demo distinta. Al final se limpia exclusivamente el contenedor creado por TemporaryDirectory. No se usan carpetas personales ni rutas obtenidas de un manifiesto externo.

In [ ]:
temporal = TemporaryDirectory(prefix="curso_copias_")
base = Path(temporal.name)
def caso(numero):
    return crear_demo(base / f"ejercicio_{numero:02d}")
print("Contenedor propio:", base)


## Ejercicio 01 · Inventario y alcance

Crea el caso 1. Obtén su inventario. Explica por qué vacio.txt está incluido y pendiente/ no lo está.

**Pista:** Usa caso(1), inventariar y sum sobre bytes.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej01 = False


In [ ]:
if completado_ej01:
    assert len(f1)==5 and sum(f["bytes"] for f in f1)==136
    assert any(f["ruta"]=="vacio.txt" and f["bytes"]==0 for f in f1)
    assert not any(f["ruta"]=="pendiente" for f in f1)
    print("Ejercicio 01: OK")
else:
    print("Ejercicio 01: pendiente")


**Para explicar en tu respuesta:** Los archivos de cero bytes conservan significado. Este contrato no registra carpetas vacías ni sus permisos.

## Ejercicio 02 · Corrupción del mismo tamaño

Compara SHA-256 de b"ABC" y b"ABD". Comprueba igual longitud y diferente huella.

**Pista:** hashlib.sha256(...).hexdigest()

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej02 = False


In [ ]:
if completado_ej02:
    assert len(a2)==len(b2) and h2a!=h2b
    print("Ejercicio 02: OK")
else:
    print("Ejercicio 02: pendiente")


**Para explicar en tu respuesta:** El tamaño no basta para comprobar contenido. Una huella de referencia también necesita protección.

## Ejercicio 03 · RPO y tiempo de recuperación

Último estado recuperable a las 02:00, fallo a las 09:00. Calcula antigüedad. Estima 100 GiB a 100 MiB/s y añade 13 minutos.

**Pista:** Convertir GiB a MiB multiplicando por 1024.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej03 = False


In [ ]:
if completado_ej03:
    assert antiguedad3==7 and transferencia3==1024 and estimacion3==1804
    print("Ejercicio 03: OK")
else:
    print("Ejercicio 03: pendiente")


**Para explicar en tu respuesta:** Son medidas y estimaciones bajo hipótesis, no objetivos automáticamente cumplidos ni benchmark real.

## Ejercicio 04 · Simulación sin publicación

Crea caso 4. Simula crear y contrasta que copias siga vacío.

**Pista:** crear_copia(p4) usa aplicar=False por defecto.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej04 = False


In [ ]:
if completado_ej04:
    assert r4["aplicado"] is False and not list((p4/"copias").iterdir())
    print("Ejercicio 04: OK")
else:
    print("Ejercicio 04: pendiente")


**Para explicar en tu respuesta:** El plan correcto no significa que exista una generación. Debe aplicarse deliberadamente.

## Ejercicio 05 · Publicación y verificación

Crea caso 5, publica una generación y verifica sus cinco archivos.

**Pista:** Conservar la identidad devuelta en copia.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej05 = False


In [ ]:
if completado_ej05:
    assert r5["verificada"] and v5["n_archivos"]==5 and v5["bytes_totales"]==136
    print("Ejercicio 05: OK")
else:
    print("Ejercicio 05: pendiente")


**Para explicar en tu respuesta:** El motor verifica antes de publicar. La comprobación posterior es otra lectura de la generación.

## Ejercicio 06 · Restauración utilizable

Crea caso 6, copia y restaura a ensayo. Compara inventarios y lee el importe del CSV y la moneda del JSON.

**Pista:** csv.DictReader usa delimiter=";"; json.loads interpreta parámetros.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej06 = False


In [ ]:
if completado_ej06:
    assert inventariar(d6)==inventariar(p6/"origen")
    assert int(filas6[0]["importe"])==1200000 and param6["moneda"]=="EUR"
    assert s6["contenido_verificado"]
    print("Ejercicio 06: OK")
else:
    print("Ejercicio 06: pendiente")


**Para explicar en tu respuesta:** La lectura prueba una parte del uso. Un proceso real requiere criterios de aceptación más amplios y sus dependencias.

## Ejercicio 07 · Recuperar una pérdida accidental

Crea caso 7 y copia. Borra SOLO informes/resumen.md del origen de esta demo. Restaura a rescate y comprueba texto recuperado y origen aún sin archivo.

**Pista:** Guarda los bytes esperados antes de unlink.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej07 = False


In [ ]:
if completado_ej07:
    assert recuperado7.read_bytes()==esperado7 and not f7.exists()
    print("Ejercicio 07: OK")
else:
    print("Ejercicio 07: pendiente")


**Para explicar en tu respuesta:** La recuperación no sobrescribe la fuente. Incorporar lo recuperado al trabajo exige una decisión posterior.

## Ejercicio 08 · Copia alterada y destino rechazado

Crea caso 8 y copia. Cambia el primer byte de un archivo de la copia, manteniendo longitud. Captura IntegridadError al restaurar.

**Pista:** Opera únicamente sobre la generación del caso 8.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej08 = False


In [ ]:
if completado_ej08:
    assert rechazo8 and f8.stat().st_size==len(b8) and not (p8/"restauraciones/rechazado").exists()
    print("Ejercicio 08: OK")
else:
    print("Ejercicio 08: pendiente")


**Para explicar en tu respuesta:** Se comprueba el fallo y la ausencia del destino final. No se normaliza una copia corrupta actualizando su manifiesto.

## Ejercicio 09 · CLI desde otro directorio

Crea caso 9 y copia. Invoca verificar mediante sys.executable, MOTOR absoluto y proyecto absoluto, con cwd=base.

**Pista:** subprocess.run con lista, capture_output=True y text=True.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej09 = False


In [ ]:
if completado_ej09:
    assert cli9.returncode==0 and dato9["verificada"]
    print("Ejercicio 09: OK")
else:
    print("Ejercicio 09: pendiente")


**Para explicar en tu respuesta:** El resultado no depende del directorio actual. No se utiliza shell=True ni interpolación de comandos.

## Ejercicio 10 · Rutas que deben rechazarse

Ensaya ../fuera, /absoluta, C:/ruta, a\b y CON. Cuenta rechazos sin escribir archivos.

**Pista:** ruta_relativa_segura lanza IntegridadError o ContratoError.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej10 = False


In [ ]:
if completado_ej10:
    assert rechazos10==len(malas10)
    print("Ejercicio 10: OK")
else:
    print("Ejercicio 10: pendiente")


**Para explicar en tu respuesta:** La validación ocurre antes de construir destinos. La defensa supone además un árbol privado y estable.

## Ejercicio 11 · Retención revisada y aplicada

Crea caso 11 con tres generaciones. Añade una carpeta desconocida con un archivo. Simula conservar dos, comprueba tres aún presentes y aplica.

**Pista:** La práctica elimina solo una generación propia y mantiene la carpeta desconocida.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej11 = False


In [ ]:
if completado_ej11:
    assert antes11 and len(plan11["retirar"])==1
    assert sum((p11/"copias"/n).exists() for n in ids11)==2
    assert (x11/"nota.txt").read_text()=="No tocar"
    assert set(plan11["conservar"])==set(real11["conservar"])
    print("Ejercicio 11: OK")
else:
    print("Ejercicio 11: pendiente")


**Para explicar en tu respuesta:** Retención por cantidad no es retención por días. En un servicio real se requiere ensayo de recuperación y política aprobada antes de retirar estados.

## Ejercicio 12 · Retención bloqueada por corrupción

Crea caso 12 con tres generaciones. Elimina un archivo dentro de una de ellas. Intenta conservar una aplicando. Debe fallar sin retirar ninguna carpeta de generación.

**Pista:** Captura IntegridadError y comprueba todos los IDs originales.

In [ ]:
# TODO: desarrolla la actividad descrita.

completado_ej12 = False


In [ ]:
if completado_ej12:
    assert rechazo12 and all((p12/"copias"/n).exists() for n in ids12)
    print("Ejercicio 12: OK")
else:
    print("Ejercicio 12: pendiente")


**Para explicar en tu respuesta:** La planificación valida antes de borrar. No arreglar el problema quitando del inventario el archivo perdido; hay que investigar y recuperar una versión válida.

## Cierre y procedimiento de recuperación

Redacta quién decide restaurar, qué generación utiliza, dónde recupera, cómo accede a claves y herramientas, qué comprueba y quién acepta el servicio. Distingue antigüedad real, RPO objetivo, tiempo técnico medido y RTO objetivo.

Consulta GUIA_RESTAURACION.md y las fuentes de la presentación. Los ejemplos Windows, macOS y rsync no se ejecutan aquí.

In [ ]:
retirar_temporal=False
if retirar_temporal:
    temporal.cleanup()
else:
    print("Contenedor disponible durante esta sesión:",base)
